# 04 · Advanced segmentation and uncertainty

How reliable are observed differences between recorded visitor, regional, browser and other categories? This notebook computes descriptive confidence intervals with explicit denominators; it does not invent demographic labels for numeric codes.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display
from scripts.advanced_segmentation import prepare, summarize, wilson_interval

ROOT = Path.cwd()
SOURCE = ROOT / "data/raw/online_shoppers_intention.csv"
if not SOURCE.is_file():
    raise FileNotFoundError(f"Missing raw source: {SOURCE}")
sessions = prepare(pd.read_csv(SOURCE))
print(f"Sessions: {len(sessions):,}; purchase outcomes: {sessions['purchased'].sum():,}")

## Outcome composition and uncertainty

Donut charts communicate **shares of recorded sessions**. Conversion comparisons instead require an outcome numerator and session denominator.

In [ ]:
purchases = int(sessions['purchased'].sum())
low, high = wilson_interval(purchases, len(sessions))
print(f"Conversion = {purchases/len(sessions):.2%}; 95% Wilson interval {low:.2f}% to {high:.2f}%")
region = summarize(sessions, 'Region')
display(region.sort_values('sessions', ascending=False).head(12))
assert region.sessions.sum() == len(sessions)
assert region.purchases.sum() == purchases

## Browser, operating-system and special-day categories

Small groups are retained in tables. `SpecialDay` uses a deliberately simple zero versus nonzero-proximity categorization, not an experimentally established treatment.

In [ ]:
for label, column in [('Browser', 'Browser'), ('Operating system', 'OperatingSystems'), ('Special-day proximity', 'special_day_band')]:
    print(label)
    display(summarize(sessions, column).sort_values('sessions', ascending=False).head(12))

## Reconciled figures

The end-to-end pipeline produces charts in `outputs/advanced_segmentation/charts/` and CSV tables in `outputs/advanced_segmentation/tables/`. This notebook does not mutate or deduplicate source records.

In [ ]:
from IPython.display import Image, display as show
fig = ROOT / 'outputs/advanced_segmentation/charts/region_intervals.png'
if not fig.is_file():
    raise FileNotFoundError('Run python -m scripts.run_upgrade before executing the notebooks')
show(Image(filename=str(fig), width=750))

**Interpretation:** Wilson intervals assume independent binomial sessions. The dataset lacks identifiers for independent validation, and measured session differences are associations rather than proven effects.